# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a nutritious, convenient fruit with several benefits and a few considerations.
    
    Key points:
    - Nutrients: ~105 calories, 3 g fiber, 12 g sugar, 422 mg potassium, vitamin C, vitamin B6.
    - Benefits: supports heart and kidney health, aids digestion, provides steady energy (carbs), convenient pre/post-workout.
    - Considerations: higher sugar content than some fruits; not ideal for very low-carb or ketose diets; people with kidney disease may need potassium limits.
    - Tips: pair with protein/fiber (e.g., peanut butter or yogurt) to stabilize appetite; ripe bananas are sweetest and easier to digest.
    
    Overall: healthy in moderation as part of a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy when eaten as part of a balanced diet.

Key benefits:
- Good source of potassium, vitamin B6, vitamin C, and fiber
- Low in fat and cholesterol-free
- Provide quick energy from natural sugars; unripe bananas have more resistant starch (prebiotic)

What to consider:
- Higher sugar and carbs as they ripen; portion if you’re watching sugar intake or have diabetes
- Whole bananas are better than juice (fiber and satiety much lower in juice)

Typical guidance:
- 1–2 bananas per day fits many diets; adjust for your total calories and goals.

If you have specific conditions (diabetes, kidney disease), tailor portions accordingly.

_Good Job!_